In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.calibration import CalibratedClassifierCV
import joblib

In [2]:
df = pd.read_csv('/Users/cemrebalci/Desktop/PhishAnalyzer/model/PhiUSIIL_Phishing_URL_Dataset.csv')
print("Toplam satır:", len(df))
print(df['label'].value_counts())

Toplam satır: 235795
label
1    134850
0    100945
Name: count, dtype: int64


In [3]:
# Ekstra güvenli URL'ler ekleyerek modeli dengele
extra_safe_urls = [
    'https://eksisozluk.com',
    'https://eksisozluk.com/entry/123',
    'https://hepsiburada.com',
    'https://trendyol.com',
    'https://sahibinden.com',
    'https://hurriyet.com.tr',
    'https://milliyet.com.tr',
    'https://ntv.com.tr',
    'https://cnnturk.com',
    'https://sozluk.gov.tr',
    'https://turkiye.gov.tr',
    'https://itu.edu.tr',
    'https://metu.edu.tr',
    'https://boun.edu.tr',
    'https://hacettepe.edu.tr',
    'https://stackoverflow.com',
    'https://medium.com',
    'https://twitter.com',
    'https://linkedin.com',
    'https://instagram.com',
    'https://twitch.tv',
    'https://spotify.com',
    'https://netflix.com',
    'https://zoom.us',
    'https://slack.com',
]

extra_rows = []
for url in extra_safe_urls:
    extra_rows.append({
        'URL': url,
        'URLLength': len(url),
        'IsHTTPS': 1,
        'NoOfSubDomain': url.count('.') - 1,
        'IsDomainIP': 0,
        'HasObfuscation': 0,
        'NoOfObfuscatedChar': 0,
        'HasPasswordField': 0,
        'Bank': 0,
        'Pay': 0,
        'Crypto': 0,
        'DegitRatioInURL': sum(c.isdigit() for c in url) / len(url),
        'NoOfAmpersandInURL': 0,
        'URLSimilarityIndex': 5.0,
        'TLDLegitimateProb': 0.9,
        'label': 1  # güvenli
    })

extra_df = pd.DataFrame(extra_rows)
df_extended = pd.concat([df, extra_df], ignore_index=True)
print("Genişletilmiş dataset:", len(df_extended))

Genişletilmiş dataset: 235820


In [4]:
features = [
    'URLLength', 'IsHTTPS', 'NoOfSubDomain', 'IsDomainIP',
    'URLSimilarityIndex', 'TLDLegitimateProb', 'HasObfuscation',
    'NoOfObfuscatedChar', 'HasPasswordField', 'Bank', 'Pay',
    'Crypto', 'DegitRatioInURL', 'NoOfAmpersandInURL'
]

X = df_extended[features]
y = df_extended['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Eğitim seti:", len(X_train))
print("Test seti:", len(X_test))

Eğitim seti: 188656
Test seti: 47164


In [5]:
# CalibratedClassifierCV gerçekçi olasılık değerleri üretir
base_model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    max_depth=20,        # Sınırsız derinlik yerine 20 — daha gerçekçi sonuçlar
    min_samples_leaf=5   # Daha az ezberleme
)

model_calibrated = CalibratedClassifierCV(base_model, cv=3, method='isotonic')
model_calibrated.fit(X_train, y_train)

y_pred = model_calibrated.predict(X_test)
print("Doğruluk:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=['Phishing', 'Güvenli']))

Doğruluk: 0.9999151895513527
              precision    recall  f1-score   support

    Phishing       1.00      1.00      1.00     20118
     Güvenli       1.00      1.00      1.00     27046

    accuracy                           1.00     47164
   macro avg       1.00      1.00      1.00     47164
weighted avg       1.00      1.00      1.00     47164



In [7]:
def test_url(url):
    import re
    from urllib.parse import urlparse
    
    parsed = urlparse(url)
    
    def get_tld_prob(u):
        safe_tlds = ['.com', '.org', '.net', '.edu', '.gov']
        risky_tlds = ['.xyz', '.tk', '.ml', '.ga', '.cf']
        for tld in safe_tlds:
            if u.endswith(tld) or tld + '/' in u:
                return 0.9
        for tld in risky_tlds:
            if tld in u:
                return 0.1
        return 0.5

    feats = {
        'URLLength': len(url),
        'IsHTTPS': 1 if url.startswith('https') else 0,
        'NoOfSubDomain': url.count('.') - 1,
        'IsDomainIP': 1 if re.match(r'\d+\.\d+\.\d+\.\d+', parsed.netloc) else 0,
        'URLSimilarityIndex': 10.0,
        'TLDLegitimateProb': get_tld_prob(url),
        'HasObfuscation': 1 if '%' in url else 0,
        'NoOfObfuscatedChar': url.count('%'),
        'HasPasswordField': 1 if 'password' in url.lower() else 0,
        'Bank': 1 if 'bank' in url.lower() else 0,
        'Pay': 1 if 'pay' in url.lower() else 0,
        'Crypto': 1 if 'crypto' in url.lower() else 0,
        'DegitRatioInURL': sum(c.isdigit() for c in url) / len(url),
        'NoOfAmpersandInURL': url.count('&'),
    }
    
    # features listesindeki sıraya göre DataFrame oluştur
    X = pd.DataFrame([feats])[features]
    pred = model_calibrated.predict(X)[0]
    prob = model_calibrated.predict_proba(X)[0]
    
    print(f"URL: {url}")
    print(f"Karar: {'GÜVENLİ ✅' if pred == 1 else 'PHİSHİNG ⚠️'}")
    print(f"Güvenli olasılığı: %{prob[1]*100:.1f}")
    print(f"Phishing olasılığı: %{prob[0]*100:.1f}")
    print()

# Test et
test_url('https://eksisozluk.com')
test_url('https://www.google.com')
test_url('http://paypa1-secure-login.xyz/verify-account')
test_url('https://hepsiburada.com')
test_url('https://trendyol.com')

URL: https://eksisozluk.com
Karar: GÜVENLİ ✅
Güvenli olasılığı: %85.0
Phishing olasılığı: %15.0

URL: https://www.google.com
Karar: PHİSHİNG ⚠️
Güvenli olasılığı: %35.0
Phishing olasılığı: %65.0

URL: http://paypa1-secure-login.xyz/verify-account
Karar: PHİSHİNG ⚠️
Güvenli olasılığı: %0.0
Phishing olasılığı: %100.0

URL: https://hepsiburada.com
Karar: GÜVENLİ ✅
Güvenli olasılığı: %81.4
Phishing olasılığı: %18.6

URL: https://trendyol.com
Karar: GÜVENLİ ✅
Güvenli olasılığı: %87.1
Phishing olasılığı: %12.9



In [8]:
extra_safe_urls = [
    'https://eksisozluk.com',
    'https://eksisozluk.com/entry/123',
    'https://hepsiburada.com',
    'https://trendyol.com',
    'https://sahibinden.com',
    'https://hurriyet.com.tr',
    'https://milliyet.com.tr',
    'https://ntv.com.tr',
    'https://cnnturk.com',
    'https://sozluk.gov.tr',
    'https://turkiye.gov.tr',
    'https://itu.edu.tr',
    'https://metu.edu.tr',
    'https://stackoverflow.com',
    'https://medium.com',
    'https://twitter.com',
    'https://linkedin.com',
    'https://instagram.com',
    'https://twitch.tv',
    'https://spotify.com',
    'https://netflix.com',
    'https://zoom.us',
    'https://slack.com',
    # Google tüm halleriyle
    'https://www.google.com',
    'https://google.com',
    'https://www.google.com/search?q=test',
    'https://mail.google.com',
    'https://drive.google.com',
    'https://docs.google.com',
    'https://www.youtube.com',
    'https://youtube.com',
    'https://www.facebook.com',
    'https://www.amazon.com',
    'https://www.microsoft.com',
    'https://www.apple.com',
    'https://github.com',
    'https://www.github.com',
    'https://wikipedia.org',
    'https://www.wikipedia.org',
    'https://reddit.com',
    'https://www.reddit.com',
]

extra_rows = []
for url in extra_safe_urls:
    extra_rows.append({
        'URL': url,
        'URLLength': len(url),
        'IsHTTPS': 1,
        'NoOfSubDomain': url.count('.') - 1,
        'IsDomainIP': 0,
        'HasObfuscation': 0,
        'NoOfObfuscatedChar': 0,
        'HasPasswordField': 0,
        'Bank': 0,
        'Pay': 0,
        'Crypto': 0,
        'DegitRatioInURL': sum(c.isdigit() for c in url) / len(url),
        'NoOfAmpersandInURL': 0,
        'URLSimilarityIndex': 5.0,
        'TLDLegitimateProb': 0.9,
        'label': 1
    })

extra_df = pd.DataFrame(extra_rows)
df_extended = pd.concat([df, extra_df], ignore_index=True)
print("Genişletilmiş dataset:", len(df_extended))

Genişletilmiş dataset: 235836


In [9]:
features = [
    'URLLength', 'IsHTTPS', 'NoOfSubDomain', 'IsDomainIP',
    'URLSimilarityIndex', 'TLDLegitimateProb', 'HasObfuscation',
    'NoOfObfuscatedChar', 'HasPasswordField', 'Bank', 'Pay',
    'Crypto', 'DegitRatioInURL', 'NoOfAmpersandInURL'
]

X = df_extended[features]
y = df_extended['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Eğitim seti:", len(X_train))
print("Test seti:", len(X_test))

Eğitim seti: 188668
Test seti: 47168


In [10]:
# CalibratedClassifierCV gerçekçi olasılık değerleri üretir
base_model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    max_depth=20,        # Sınırsız derinlik yerine 20 — daha gerçekçi sonuçlar
    min_samples_leaf=5   # Daha az ezberleme
)

model_calibrated = CalibratedClassifierCV(base_model, cv=3, method='isotonic')
model_calibrated.fit(X_train, y_train)

y_pred = model_calibrated.predict(X_test)
print("Doğruluk:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=['Phishing', 'Güvenli']))

Doğruluk: 0.999915196743555
              precision    recall  f1-score   support

    Phishing       1.00      1.00      1.00     20088
     Güvenli       1.00      1.00      1.00     27080

    accuracy                           1.00     47168
   macro avg       1.00      1.00      1.00     47168
weighted avg       1.00      1.00      1.00     47168



In [11]:
def test_url(url):
    import re
    from urllib.parse import urlparse
    
    parsed = urlparse(url)
    
    def get_tld_prob(u):
        safe_tlds = ['.com', '.org', '.net', '.edu', '.gov']
        risky_tlds = ['.xyz', '.tk', '.ml', '.ga', '.cf']
        for tld in safe_tlds:
            if u.endswith(tld) or tld + '/' in u:
                return 0.9
        for tld in risky_tlds:
            if tld in u:
                return 0.1
        return 0.5

    feats = {
        'URLLength': len(url),
        'IsHTTPS': 1 if url.startswith('https') else 0,
        'NoOfSubDomain': url.count('.') - 1,
        'IsDomainIP': 1 if re.match(r'\d+\.\d+\.\d+\.\d+', parsed.netloc) else 0,
        'URLSimilarityIndex': 10.0,
        'TLDLegitimateProb': get_tld_prob(url),
        'HasObfuscation': 1 if '%' in url else 0,
        'NoOfObfuscatedChar': url.count('%'),
        'HasPasswordField': 1 if 'password' in url.lower() else 0,
        'Bank': 1 if 'bank' in url.lower() else 0,
        'Pay': 1 if 'pay' in url.lower() else 0,
        'Crypto': 1 if 'crypto' in url.lower() else 0,
        'DegitRatioInURL': sum(c.isdigit() for c in url) / len(url),
        'NoOfAmpersandInURL': url.count('&'),
    }
    
    # features listesindeki sıraya göre DataFrame oluştur
    X = pd.DataFrame([feats])[features]
    pred = model_calibrated.predict(X)[0]
    prob = model_calibrated.predict_proba(X)[0]
    
    print(f"URL: {url}")
    print(f"Karar: {'GÜVENLİ ✅' if pred == 1 else 'PHİSHİNG ⚠️'}")
    print(f"Güvenli olasılığı: %{prob[1]*100:.1f}")
    print(f"Phishing olasılığı: %{prob[0]*100:.1f}")
    print()

# Test et
test_url('https://eksisozluk.com')
test_url('https://www.google.com')
test_url('http://paypa1-secure-login.xyz/verify-account')
test_url('https://hepsiburada.com')
test_url('https://trendyol.com')

URL: https://eksisozluk.com
Karar: GÜVENLİ ✅
Güvenli olasılığı: %73.0
Phishing olasılığı: %27.0

URL: https://www.google.com
Karar: GÜVENLİ ✅
Güvenli olasılığı: %88.6
Phishing olasılığı: %11.4

URL: http://paypa1-secure-login.xyz/verify-account
Karar: PHİSHİNG ⚠️
Güvenli olasılığı: %0.0
Phishing olasılığı: %100.0

URL: https://hepsiburada.com
Karar: GÜVENLİ ✅
Güvenli olasılığı: %70.1
Phishing olasılığı: %29.9

URL: https://trendyol.com
Karar: GÜVENLİ ✅
Güvenli olasılığı: %79.3
Phishing olasılığı: %20.7



In [12]:
joblib.dump(model_calibrated, '/Users/cemrebalci/Desktop/PhishAnalyzer/model/phishanalyzer_model.pkl')
joblib.dump(features, '/Users/cemrebalci/Desktop/PhishAnalyzer/model/phishanalyzer_features.pkl')
print("✅ Model kaydedildi!")

✅ Model kaydedildi!


In [13]:
# Önce Railway'deki versiyonu kontrol edelim
import sklearn
print(sklearn.__version__)

1.7.2
